In [ ]:
import shap
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # prevents display issues in VS Code
import warnings
warnings.filterwarnings('ignore')

# Load best model (XGBoost regularised)
model = joblib.load('../models/best_model.pkl')

# Load test data
data = np.load('../data/processed/train_test_splits.npz', allow_pickle=True)
X_test = data['X_test']
y_test = data['y_test']
feature_names = data['feature_names'].tolist()

# Convert to DataFrame with feature names
X_test_df = pd.DataFrame(X_test, columns=feature_names)

print(f"Model loaded: XGBoost (regularised)")
print(f"Test set shape: {X_test_df.shape}")
print(f"Features: {feature_names}")
print(f"\nClass distribution in test set:")
print(f"  Good credit (0): {(y_test==0).sum()}")
print(f"  Bad credit (1):  {(y_test==1).sum()}")

In [ ]:
# Compute SHAP values using TreeExplainer
# TreeExplainer is the most efficient method for XGBoost
print("Computing SHAP values...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_df)

print(f"SHAP values computed!")
print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected value (base rate): {explainer.expected_value:.4f}")
print(f"\nTop 5 most important features (mean |SHAP|):")
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feature_importance = pd.Series(mean_abs_shap, index=feature_names)
top5 = feature_importance.sort_values(ascending=False).head(5)
for feat, val in top5.items():
    print(f"  {feat:<25} {val:.4f}")

In [ ]:
# Verify SHAP values sum to prediction
import numpy as np

# Check for first 3 test samples
print("SHAP SANITY CHECK:")
print("=" * 60)
for i in range(3):
    shap_sum = shap_values[i].sum() + explainer.expected_value
    model_pred = model.predict_proba(X_test_df.iloc[[i]])[0][1]
    print(f"Sample {i+1}:")
    print(f"  SHAP sum + base: {shap_sum:.4f}")
    print(f"  Model predict:   {model_pred:.4f}")
    print(f"  Actual label:    {y_test[i]} ({'Bad' if y_test[i]==1 else 'Good'})")
    print()

In [ ]:
# Find a bad credit applicant the model correctly predicted
bad_indices = np.where(y_test == 1)[0]
model_preds = model.predict(X_test_df)

# Find one the model got right (correctly predicted bad)
correct_bad = [i for i in bad_indices if model_preds[i] == 1]
print(f"Bad credit applicants in test: {len(bad_indices)}")
print(f"Correctly predicted bad: {len(correct_bad)}")

# Pick the one with highest SHAP magnitude (most interesting)
shap_magnitudes = [np.abs(shap_values[i]).sum() for i in correct_bad]
best_example_idx = correct_bad[np.argmax(shap_magnitudes)]

print(f"\nSelected example index: {best_example_idx}")
print(f"Actual label: {y_test[best_example_idx]} (Bad credit)")
print(f"Model prediction: {model_preds[best_example_idx]} (Bad credit)")
print(f"\nApplicant profile:")
for feat, val in X_test_df.iloc[best_example_idx].items():
    print(f"  {feat:<25} {val}")

In [ ]:
# Generate SHAP waterfall plot
fig, ax = plt.subplots(figsize=(10, 8))

# Create explanation object for this sample
explanation = shap.Explanation(
    values=shap_values[best_example_idx],
    base_values=explainer.expected_value,
    data=X_test_df.iloc[best_example_idx].values,
    feature_names=feature_names
)

shap.plots.waterfall(explanation, max_display=15, show=False)

plt.title('Chart 1: Why this applicant was classified as Bad Credit Risk\n'
          '(Red bars increase risk, Blue bars decrease risk)',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../shap_plots/shap_waterfall.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: shap_plots/shap_waterfall.png")
print("This is Chart 1 for your MS Forms questionnaire")

In [ ]:
# Generate SHAP beeswarm plot — global feature importance
fig, ax = plt.subplots(figsize=(10, 8))

explanation_all = shap.Explanation(
    values=shap_values,
    base_values=explainer.expected_value,
    data=X_test_df.values,
    feature_names=feature_names
)

shap.plots.beeswarm(explanation_all, max_display=15, show=False)

plt.title('Chart 2: Which factors matter most for credit risk decisions\n'
          '(Features ranked by importance — dots show all 200 applicants)',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../shap_plots/shap_beeswarm.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: shap_plots/shap_beeswarm.png")
print("This is Chart 2 for your MS Forms questionnaire")

In [ ]:
# Bar plot — mean absolute SHAP values (cleaner for non-technical participants)
fig, ax = plt.subplots(figsize=(10, 7))

mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_names
).sort_values(ascending=True)

colors = ['#C0392B' if v > mean_shap.mean() else '#2E8BC0'
          for v in mean_shap.values]

bars = ax.barh(mean_shap.index, mean_shap.values, color=colors)
ax.axvline(mean_shap.mean(), color='gray', linestyle='--',
           alpha=0.7, label='Average importance')
ax.set_xlabel('Average impact on credit risk prediction', fontsize=12)
ax.set_title('Figure 10: Feature Importance — Average SHAP Values\n'
             'Red = above average importance', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../shap_plots/shap_bar_summary.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: shap_plots/shap_bar_summary.png")

In [ ]:
# Force plot for bad credit applicant
shap.initjs()

# Use the same bad credit example from Day 2
force_bad = shap.force_plot(
    explainer.expected_value,
    shap_values[best_example_idx],
    X_test_df.iloc[best_example_idx],
    feature_names=feature_names,
    matplotlib=True,
    show=False,
    figsize=(16, 4)
)

plt.title('Chart 3a: Decision breakdown — Bad Credit Risk Applicant\n'
          'Red pushes toward bad risk, Blue pushes toward good risk',
          fontsize=12, fontweight='bold', y=1.15)
plt.savefig('../shap_plots/shap_force_bad.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: shap_plots/shap_force_bad.png")

In [ ]:
# Find a correctly predicted GOOD credit applicant
good_indices = np.where(y_test == 0)[0]
correct_good = [i for i in good_indices if model_preds[i] == 0]

# Pick one with clearest SHAP signal
shap_mag_good = [np.abs(shap_values[i]).sum() for i in correct_good]
good_example_idx = correct_good[np.argmax(shap_mag_good)]

# Force plot for good credit applicant
shap.force_plot(
    explainer.expected_value,
    shap_values[good_example_idx],
    X_test_df.iloc[good_example_idx],
    feature_names=feature_names,
    matplotlib=True,
    show=False,
    figsize=(16, 4)
)

plt.title('Chart 3b: Decision breakdown — Good Credit Risk Applicant\n'
          'Blue pushes toward good risk, Red pushes toward bad risk',
          fontsize=12, fontweight='bold', y=1.15)
plt.savefig('../shap_plots/shap_force_good.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: shap_plots/shap_force_good.png")
print("\nAll 3 chart types generated successfully!")

In [ ]:
import os

print("=" * 50)
print("ALL SHAP PLOTS SAVED")
print("=" * 50)

plots = [
    ('shap_waterfall.png',   'Chart 1 — Waterfall (bad credit)'),
    ('shap_beeswarm.png',    'Chart 2 — Beeswarm (all applicants)'),
    ('shap_bar_summary.png', 'Figure 10 — Bar summary'),
    ('shap_force_bad.png',   'Chart 3a — Force (bad credit)'),
    ('shap_force_good.png',  'Chart 3b — Force (good credit)'),
]

for fname, desc in plots:
    path = f'../shap_plots/{fname}'
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = f"✓ {size//1024}KB" if exists else "✗ MISSING"
    print(f"  {status:<12} {fname:<30} {desc}")

print("\nCharts for MS Forms questionnaire:")
print("  Chart 1: shap_waterfall.png")
print("  Chart 2: shap_beeswarm.png")
print("  Chart 3: shap_force_bad.png + shap_force_good.png")